# Travel-Time Surface — Single Destination

A **travel-time surface** answers: *"From every point in this city, how
long does it take to reach this clinic?"*

The output is a raster where each pixel value is the travel time in minutes
to the destination.  This is the building block for:
- Catchment population analysis (who is within 30/60 min?)
- Facility siting optimisation
- Health access inequality mapping

**What's covered**
1. Constructing a `BoundingBox` and choosing resolution
2. Computing a surface with `OsmnxSurfaceRouter`
3. Inspecting `TravelTimeSurface` — shape, stats, pixel coordinates
4. Threshold masks: "reachable within 30 minutes"
5. Saving and reloading from GeoTIFF
6. Interactive map: folium choropleth
7. Static publication figure: matplotlib


## Setup

In [ ]:
from idd_travel_time.access import (
    BoundingBox, OsmnxSurfaceRouter, TravelTimeSurface
)
from idd_travel_time.constants import TransportMode
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Destination: Kenyatta National Hospital, Nairobi
DESTINATION = (-1.3006, 36.8073)   # (lat, lon)

# Bounding box for Nairobi city centre ~10×10 km
BBOX = BoundingBox(west=36.72, south=-1.37, east=36.90, north=-1.22)

# 0.01° ≈ 1.1 km at equator — good for a first run
RESOLUTION = 0.01

print(f"Bounding box: {BBOX.width_deg:.2f}° × {BBOX.height_deg:.2f}°")
nrows = int(BBOX.height_deg / RESOLUTION)
ncols = int(BBOX.width_deg / RESOLUTION)
print(f"Grid size at {RESOLUTION}°: {nrows} rows × {ncols} cols = {nrows*ncols:,} pixels")


## 1 · Compute the surface

In [ ]:
router = OsmnxSurfaceRouter()

print("Computing travel-time surface (downloads OSM graph on first run) …")
surface = router.compute_surface(
    destination=DESTINATION,
    bbox=BBOX,
    resolution_deg=RESOLUTION,
    mode=TransportMode.WALKING,
)

print(surface)


## 2 · Inspect the surface

In [ ]:
stats = surface.summary_stats()
print(f"Grid:          {surface.nrows} rows × {surface.ncols} cols")
print(f"Resolution:    {surface.resolution_deg:.4f}° (~{surface.resolution_deg * 111:.0f} m at equator)")
print(f"Valid pixels:  {stats['count']:,}  ({1 - stats['nan_fraction']:.0%} of grid)")
print(f"Travel times:  {stats['min']:.1f} – {stats['max']:.1f} min  (mean {stats['mean']:.1f} min)")
print()
print(f"Coverage within 30 min: {surface.coverage_at(30):.1%}")
print(f"Coverage within 60 min: {surface.coverage_at(60):.1%}")


## 3 · Threshold mask

In [ ]:
reachable_30 = surface.threshold(30.0)   # boolean array
reachable_60 = surface.threshold(60.0)

print(f"Pixels within 30 min: {reachable_30.sum():,} ({reachable_30.mean():.1%})")
print(f"Pixels within 60 min: {reachable_60.sum():,} ({reachable_60.mean():.1%})")


## 4 · Static map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Continuous travel time
ax = axes[0]
im = ax.imshow(
    surface.data, cmap="RdYlGn_r", vmin=0, vmax=60,
    extent=[surface.bbox.west, surface.bbox.east,
            surface.bbox.south, surface.bbox.north],
    origin="upper", aspect="auto",
)
ax.scatter([DESTINATION[1]], [DESTINATION[0]], c="blue", s=120,
           zorder=5, label="KNH (destination)")
plt.colorbar(im, ax=ax, label="Travel time (min)")
ax.set_title("Walking travel time to KNH")
ax.legend()

# 30-min catchment
ax = axes[1]
cmap = mcolors.ListedColormap(["#d62728", "#2ca02c"])
ax.imshow(
    reachable_30.astype(np.float32), cmap=cmap, vmin=0, vmax=1,
    extent=[surface.bbox.west, surface.bbox.east,
            surface.bbox.south, surface.bbox.north],
    origin="upper", aspect="auto",
)
ax.scatter([DESTINATION[1]], [DESTINATION[0]], c="blue", s=120, zorder=5)
ax.set_title("Within 30 min walking (green = yes)")

plt.tight_layout()
plt.show()


## 5 · Save and reload

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    p = surface.to_geotiff(Path(tmpdir) / "nairobi_walking.tif")
    print(f"Saved: {p}  ({p.stat().st_size // 1024} KB)")

    loaded = TravelTimeSurface.from_geotiff(p)
    print(f"Loaded: {loaded}")
    print(f"Backend preserved: {loaded.backend!r}")
    print(f"Destination preserved: {loaded.destination}")


## 6 · Interactive folium map

In [ ]:
import folium
import json

# Convert surface to a GeoJSON choropleth at coarser resolution for speed
gdf = surface.resample(0.02).to_geodataframe()
gdf = gdf[gdf["travel_time_min"] <= 90]   # clip outliers

m = folium.Map(location=list(DESTINATION), zoom_start=12, tiles="CartoDB positron")

folium.GeoJson(
    gdf.__geo_interface__,
    name="Travel time (min)",
    style_function=lambda f: {
        "fillColor": _time_to_color(f["properties"]["travel_time_min"]),
        "color": "none",
        "fillOpacity": 0.6,
        "radius": 8,
    },
).add_to(m)

folium.Marker(
    location=list(DESTINATION),
    popup="Kenyatta National Hospital",
    icon=folium.Icon(color="blue"),
).add_to(m)

def _time_to_color(t):
    if t < 15: return "#1a9641"
    if t < 30: return "#a6d96a"
    if t < 45: return "#fdae61"
    return "#d7191c"

m


## 7 · Next steps

- **Higher resolution:** try `resolution_deg=0.001` (~111 m) for neighbourhood-level detail
- **Different modes:** replace `WALKING` with `DRIVING` or `TRANSIT` (with a `TransitProfile`)
- **Multiple destinations:** see notebook `12_ensemble_surfaces.ipynb`
- **Compare backends:** see notebook `11_surface_backends.ipynb`
